In [25]:
import matplotlib.pyplot as plt
import scipy.stats as stats
import pandas as pd
import numpy as np
import re
from glob import glob
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

In [26]:
files = glob("results/*")

In [27]:
def read_pair_file(filename):
    pairs = {}
    with open(filename, 'r') as f:
        for line in f:
            match = re.match(r'#pair\s+"(.+?)"\s+"(.+?)"', line)
            if match:
                key, value = match.group(1), match.group(2)
                # Try to convert to numeric where possible
                try:
                    value = pd.to_numeric(value)
                except (ValueError, TypeError):
                    pass
                pairs[key] = value
    return pd.Series(pairs)

In [31]:
results = []
for filename in files:
    s = read_pair_file(filename)
    words = filename.split("/")[1].split("_")
    if len(words) < 5:
        continue
    s['algorithm'] = words[0]
    s['heuristic'] = words[1]
    s['weight'] = float(words[2])
    s['width'] = int(words[3])
    s['instance'] = int(words[4])
    if (s['algorithm'] in ["rrd", "ees"]) and (s['width'] != -1):
        continue
    results.append(s)

        

In [63]:
res1 = pd.DataFrame(results)

In [65]:
res1.loc[res1.weight == -1, "weight"] = 1

res1.loc[res1["total raw cpu time"].isna(),"total raw cpu time" ] = 300
res1.loc[res1['final sol cost'].isna(),'final sol cost' ] = res1['final sol cost'].max()



In [34]:
res.columns

Index(['wall start date', 'wall start time', 'machine id', 'cost',
       'initial heuristic', 'initial distance', 'algorithm', 'final sol cost',
       'state size', 'packed state size', 'final sol length',
       'total raw cpu time', 'total wall time', 'total nodes expanded',
       'total nodes generated', 'total nodes duplicated',
       'total nodes reopened', 'closed list type', 'closed fill',
       'closed collisions', 'closed resizes', 'closed buckets',
       'closed max bucket fill', 'open list type', 'node size', 'weight',
       'h error last', 'd error last', 'width', 'Astar expansions',
       'beam expansions', 'wall finish time', 'wall finish date',
       'max virtual kilobytes', 'heuristic', 'instance'],
      dtype='object')

In [55]:
for heur in set(res1.heuristic):
    res = res1.loc[res1.heuristic == heur]
    fig = go.Figure()
    for (width, algorithm), g1 in res.groupby(['width', 'algorithm']):
        xs = []
        ys = []
        for weight, group in g1.groupby('weight'):
            group = group.set_index('instance').sort_index()
            xs.append(weight)
            ys.append(group['total raw cpu time'].sum())
        
        xs, ys = zip(*sorted(zip(xs, ys)))
        fig.add_trace(go.Scatter(
            x=list(xs),
            y=list(ys),
            mode='lines+markers',
            name=f"{algorithm} width={width}",
        ))
    fig.update_layout(
        title='Korf 100 - ' + heur,
        xaxis_title='Weight',
        yaxis_title='total CPU time (s)',
            yaxis_type='log',
        xaxis_type='log',
        yaxis=dict(tickformat='~g'),
        xaxis=dict(tickformat='~g'),
    )
            
    fig.show()
    plotly.offline.plot(fig, filename=heur + '-runtime.html')

In [56]:
for heur in set(res1.heuristic):
    res = res1.loc[res1.heuristic == heur]
    fig = go.Figure()
    for (width, algorithm), g1 in res.groupby(['width', 'algorithm']):
        xs = []
        ys = []
        for weight, group in g1.groupby('weight'):
            group = group.set_index('instance').sort_index()
            xs.append(weight)
            ys.append((group['total raw cpu time'] < 500.0).sum())
        
        xs, ys = zip(*sorted(zip(xs, ys)))
        fig.add_trace(go.Scatter(
            x=list(xs),
            y=list(ys),
            mode='lines+markers',
            name=f"{algorithm} width={width}",
        ))
    fig.update_layout(
        title='Korf 100 - ' + heur,
        xaxis_title='Weight',
        yaxis_title='Percent Solved',
            yaxis_type='log',
        xaxis_type='log',
        yaxis=dict(tickformat='~g'),
        xaxis=dict(tickformat='~g'),
    )
            
    fig.show()
    plotly.offline.plot(fig, filename=heur + '-percsolved.html')

In [67]:
for heur in set(res1.heuristic):
    res = res1.loc[res1.heuristic == heur]
    fig = go.Figure()
    for (width, algorithm), g1 in res.groupby(['width', 'algorithm']):
        xs = []
        ys = []
        for weight, group in g1.groupby('weight'):
            group = group.set_index('instance').sort_index()
            xs.append(weight)
            ys.append(group['final sol cost'].sum())
        
        xs, ys = zip(*sorted(zip(xs, ys)))
        fig.add_trace(go.Scatter(
            x=list(xs),
            y=list(ys),
            mode='lines+markers',
            name=f"{algorithm} width={width}",
        ))
    fig.update_layout(
        title='Korf 100 - ' + heur,
        xaxis_title='Weight',
        yaxis_title='Total Cost *With penalized timeout*',
            yaxis_type='log',
        xaxis_type='log',
        yaxis=dict(tickformat='~g'),
        xaxis=dict(tickformat='~g'),
    )
            
    fig.show()
    plotly.offline.plot(fig, filename=heur + '-total-cost.html')

In [54]:
plotly.offline.plot(fig, filename='results.html')

'results.html'

In [61]:
res1['final sol cost'].isna().any()

np.True_